# Web Scraping con Selenium

Selenium es un *framework* de automatización de navegadores. Su uso principal es la realización de pruebas en aplicaciones web, ya que se puede automatizar una serie de interacciones del usuario y comprobar que los resultados son los esperados. Sin embargo, por esa misma razón es un *framework* que permite hacer un *web scraping* muy preciso, simulando interacciones muy concretas de un usuario: rellenar formularios, pulsar botones y otros elementos, etc.

Para utilizar Selenium en Python es necesario:
1. Instalar el paquete Python con `pip install selenium`
2. Tener instalado en el sistema el navegador que vamos a usar (Firefox, Chrome, Edge o Safari)
3. Descargar el controlador del navegador que vamos a automatizar. Por ejemplo el de Chrome está en https://chromedriver.chromium.org/downloads y el de Firefox en https://github.com/mozilla/geckodriver/releases. Más información en https://selenium-python.readthedocs.io/installation.html#drivers

La documentación de Selenium en Python está disponible en https://selenium-python.readthedocs.io/

Una vez elegido el navegador a utilizar, cargaremos el controlador de la siguiente manera:
    
    driver = webdriver.Firefox()  # Para Firefox
    driver = webdriver.Chrome()   # Para Chrome

## Selección de elementos

El proceso de automatizar la navegación de una página web consiste en seleccionar elementos de la página y realizar una acción con ellos: escribir texto, pulsar una tecla, hacer clic, etc. Para seleccionar elementos disponemos de la función `driver.find_element` y la condición de búsqueda puede ser de distintos tipos:

* `By.ID`: búsqueda por identificador (atributo `id`)
* `By.NAME`: búsqueda por nombre (atributo `name`)
* `By.XPATH`: búsqueda usando una ruta XPath
* `By.LINK_TEXT`: encontrar un enlace con un texto concreto
* `By.PARTIAL_LINK_TEXT`: encontrar un enlace con un texto parcial
* `By.TAG_NAME`: búsqueda por tipo de etiqueta HTML
* `By.CLASS_NAME`: búsqueda por clase CSS (atributo `class`)
* `By.CSS_SELECTOR`: búsqueda usando un selector CSS

Ejemplos:

    element = driver.find_element(By.ID, "passwd-id")
    element = driver.find_element(By.CSS_SELECTOR, "input#passwd-id")

Una vez encontrado un elemento, se pueden hacer varias acciones. Por ejemplo:

    element.clear()
    element.send_keys("MADRID")
    element.send_keys("MADRID", Keys.ARROW_DOWN, Keys.RETURN)
    element.click()

## Esperar a contenido asíncrono

En muchos casos las páginas web tienen contenido que se carga de manera asíncrona de distintas fuentes, y es necesario esperar a que esté disponible antes de comenzar a navegar la página web. Para ello, Selenium permite definir esperas:

    element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "boton"))
    )

En este caso se espera a que la página tenga cargado un elemento de identificado `boton` durante un máximo de 10 segundos. En caso de que después de ese tiempo no exista ningún elemento que encaje con la búsqueda se lanzará una excepción `TimeoutException`.

También se puede configurar Selenium para que siempre se espere una cierta cantidad de tiempo (temporizador implícito) en cualquier búsqueda de elementos. Esto se hace con:

    driver.implicitly_wait(12)  # 12 segundos

## Ejemplo completo con Selenium: obtener los productos de una tienda (página sintética)

In [1]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging

logger = logging.getLogger('selenium')
logger.setLevel(logging.DEBUG)
handler = logging.FileHandler('selenium.log')
logger.addHandler(handler)

# Elegimos entre Firefox o Chrome
# from selenium.webdriver.firefox.options import Options
# opts = Options()
# opts.add_argument("--force-browser-download")
# driver = webdriver.Firefox(options=opts)
# driver = webdriver.Firefox()
driver = webdriver.Chrome()

# Cargamos la página principal
driver.get("https://practice.qabrains.com/ecommerce")

# Seleccionamos el "input" de email, esperando a que se cargue
origen = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.ID, "email")))
origen.clear()
origen.send_keys("practice@qabrains.com", Keys.RETURN)

# Seleccionamos el "input" de email, esperando a que se cargue
origen = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.ID, "password")))
origen.clear()
origen.send_keys("Password123", Keys.RETURN)

# Pulsamos el botón de envío de formulario
submit_btn = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "button.btn-submit")))
submit_btn.click()

# Esperamos a que se cargue la información de los productos
# Usamos un selector CSS para esperar que se cargue el elemento DIV principal
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div.products")))

# Recuperamos el código HTML y cerramos el navegador
html = driver.page_source
driver.close()  # Puedes dejar el navegador abierto para depurar

In [2]:
print(html[:5000])

<html lang="en" style="scroll-behavior: smooth; --questrial-font: 'Questrial', 'Questrial Fallback'; --oswald-font: 'Oswald', 'Oswald Fallback'; color-scheme: light;" data-theme="light"><head><link rel="stylesheet" href="/_next/static/css/fc88df96f5c3b850.css" data-precedence="next"><link rel="preload" as="script" fetchpriority="low" href="/_next/static/chunks/webpack-7f89f43bd35b7caf.js"><script src="/_next/static/chunks/4bd1b696-f785427dddbba9fb.js" async=""></script><script src="/_next/static/chunks/1255-d271660abf95e684.js" async=""></script><script src="/_next/static/chunks/main-app-08607c612937c180.js" async=""></script><script src="/_next/static/chunks/1311-fcfcc5fecb199a25.js" async=""></script><script src="/_next/static/chunks/8720-d7f67a5726cf715d.js" async=""></script><script src="/_next/static/chunks/5125-8626032b05185706.js" async=""></script><script src="/_next/static/chunks/4243-e1cb11d2e59a0e94.js" async=""></script><script src="/_next/static/chunks/8571-a642d85b9370c2d

In [3]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, 'html.parser')

# DIV contenedor de todos los productos
productos = soup.find("div", "products")
print(len(productos))

9


In [4]:
productos.contents[0]  # Primer hijo

<div class="flex flex-col gap-3 relative group"><span class="absolute top-3 right-3 cursor-pointer" role="button"><button class="cursor-pointer" style="color: rgb(34, 34, 34);" type="button"><svg class="w-5 h-5" fill="currentColor" height="1em" stroke="currentColor" stroke-width="2" viewbox="0 0 512 512" width="1em" xmlns="http://www.w3.org/2000/svg"><path d="M349.6 64c-36.4 0-70.7 16.7-93.6 43.9C233.1 80.7 198.8 64 162.4 64 97.9 64 48 114.2 48 179.1c0 79.5 70.7 143.3 177.8 241.7L256 448l30.2-27.2C393.3 322.4 464 258.6 464 179.1 464 114.2 414.1 64 349.6 64zm-80.8 329.3l-4.2 3.9-8.6 7.8-8.6-7.8-4.2-3.9c-50.4-46.3-94-86.3-122.7-122-28-34.7-40.4-63.1-40.4-92.2 0-22.9 8.4-43.9 23.7-59.3 15.2-15.4 36-23.8 58.6-23.8 26.1 0 52 12.2 69.1 32.5l24.5 29.1 24.5-29.1c17.1-20.4 43-32.5 69.1-32.5 22.6 0 43.4 8.4 58.7 23.8 15.3 15.4 23.7 36.5 23.7 59.3 0 29-12.5 57.5-40.4 92.2-28.8 35.7-72.3 75.7-122.8 122z"></path></svg></button></span><a class="bg-gray-200 group-hover:bg-gray-300 p-4 decoration-0 mi

In [5]:
productos.contents[0].find('a', 'block')

<a class="text-lg block decoration-0 font-semibold font-oswald text-gray-900" href="/ecommerce/product-details?id=1">Sample Shirt Name</a>

In [6]:
productos.contents[0].find('span', 'text-lg')

<span class="text-lg font-bold text-black">$49.99</span>

In [7]:
def extrae_nombre_precio(div_elem):
    """ Extrae el nombre y precio (como float) de un DIV de un producto """
    nombre_a = div_elem.find("a", "block").string
    precio_str = div_elem.find('span', 'text-lg').string
    precio = float(precio_str[1:])
    return nombre_a, precio
    
# Lista de datos extraídos de productos, ordenador por precio descendente
# Recorro los hijos DIV directos del DIV productos y ordeno el resultado
prods = [extrae_nombre_precio(elem) for elem in productos.find_all("div", recursive=False)]
sorted(prods, key = lambda x : x[1], reverse=True)

[('Sample Sunglass Name', 256.45),
 ('Sample Sunglass Name', 256.45),
 ('Sample Jacket Name', 129.5),
 ('Sample Jacket Name', 111.0),
 ('Sample Shoe Name', 89.0),
 ('Sample Trouser Name', 72.0),
 ('Sample T-Shirt Name', 56.45),
 ('Sample Shirt Name', 49.99),
 ('Sample T-Shirt Name', 45.0)]